In [9]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def homomorphic_filter(image, d0=30, rl=0.5, rh=2.0, c=1.0):
    # 1. Log Transform
    img_log = np.log1p(np.array(image, dtype="float"))
    
    # 2. DFT
    dft = np.fft.fft2(img_log)
    dft_shift = np.fft.fftshift(dft)
    
    # 3. Create High-Boost Filter (H)
    rows, cols = image.shape
    crow, ccol = rows // 2, cols // 2
    
    y, x = np.ogrid[-crow:rows-crow, -ccol:cols-ccol]
    dist_sq = x**2 + y**2
    
    # Gaussian High-Pass Filter formula for Homomorphic
    H = (rh - rl) * (1 - np.exp(-c * (dist_sq / (d0**2)))) + rl
    
    # 4. Apply Filter and Inverse DFT
    filtered_dft = dft_shift * H
    img_back = np.fft.ifftshift(filtered_dft)
    img_back = np.fft.ifft2(img_back)
    
    # 5. Exponential (Inverse Log)
    img_res = np.exp(np.real(img_back)) - 1
    
    return np.uint8(np.clip(img_res, 0, 255))

# Usage
img = cv2.imread('runway.png', 0)
result = homomorphic_filter(img)
cv2.imwrite('ques12/homomorphic_result.png', result)

True